In [ ]:
!pip install easyocr imutils gradio opencv-python-headless matplotlib pillow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 6.8 MB/s eta 0:00:00


In [ ]:
#=========================
# Cell 2 - import library
#=========================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import imutils
import easyocr
import re
from collections import deque

import zipfile
import os
import glob

import tempfile, os
from IPython.display import Image as IPImage, display as ipdisplay, clear_output

#===================
# Cell 3 - load OCR
#===================
ocr = easyocr.Reader(['en'], gpu=True)
print("\nOCR loaded")

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete
OCR loaded


In [ ]:
# =============================
# CELL 4 — State identification
# =============================

PLATE_MAP = {
    # WP Kuala Lumpur
    "W":"WP Kuala Lumpur", "WA":"WP Kuala Lumpur",
    "WB":"WP Kuala Lumpur", "WC":"WP Kuala Lumpur","V":"WP Kuala Lumpur",
    "WW":"WP Kuala Lumpur", "WWB":"WP Kuala Lumpur", "WWC":"WP Kuala Lumpur",
    # WP Putrajaya
    "F":"WP Putrajaya",
    "L":"WP Labuan",
    # Selangor
    "B":"Selangor",  "BA":"Selangor", "BB":"Selangor",
    "BC":"Selangor", "BD":"Selangor", "BE":"Selangor",
    "BF":"Selangor", "BG":"Selangor",
    # Selangor V-series
    "VA":"Selangor", "VB":"Selangor", "VC":"Selangor",
    "VD":"Selangor", "VE":"Selangor", "VF":"Selangor",
    "VG":"Selangor", "VH":"Selangor", "VJ":"Selangor",
    "VK":"Selangor", "VM":"Selangor", "VN":"Selangor",
    "VP":"Selangor", "VR":"Selangor", "VS":"Selangor",
    "VT":"Selangor", "VU":"Selangor",
    # Johor
    "J":"Johor",  "JA":"Johor", "JB":"Johor",
    "JC":"Johor", "JD":"Johor", "JE":"Johor",
    # Kedah
    "K":"Kedah",  "KA":"Kedah", "KB":"Kedah",
    "KC":"Kedah", "KD":"Kedah", "KE":"Kedah", "KF":"Kedah",
    # Perak
    "A":"Perak",  "AA":"Perak", "AB":"Perak",
    "AC":"Perak", "AD":"Perak", "AE":"Perak",
    # Pahang
    "C":"Pahang", "CA":"Pahang", "CB":"Pahang",
    # Kelantan
    "D":"Kelantan", "DA":"Kelantan", "DB":"Kelantan",
    # Melaka
    "M":"Melaka",  "MA":"Melaka", "MB":"Melaka", "MC":"Melaka",
    # Negeri Sembilan
    "N":"Negeri Sembilan", "NA":"Negeri Sembilan", "NB":"Negeri Sembilan",
    # Pulau Pinang
    "P":"Pulau Pinang", "PA":"Pulau Pinang", "PB":"Pulau Pinang",
    "PC":"Pulau Pinang", "PD":"Pulau Pinang",
    # Perlis
    "R":"Perlis",
    # Terengganu
    "T":"Terengganu", "TA":"Terengganu", "TB":"Terengganu",
    # Sabah
    "SA":"Sabah", "SB":"Sabah", "SC":"Sabah", "SD":"Sabah","SG":"Sabah", "SK":"Sabah",
    "SS":"Sabah","ST":"Sabah","SU":"Sabah","SY":"Sabah",
    # Sarawak
    "Q":"Sarawak","QA":"Sarawak", "QB":"Sarawak", "QAA":"Sarawak", "QAB":"Sarawak",
    "QC":"Sarawak","QH":"Sarawak","QK":"Sarawak","QK":"Sarawak","QM":"Sarawak",
    "QP":"Sarawak","QR":"Sarawak","QS":"Sarawak","QT":"Sarawak",
    # Special
    "ATM":"Military","Z":"Military",
    "CD":"Diplomatic Corps", "CC":"Diplomatic Corps",
    "KDN":"Police", "VIP":"VIP / Special", "BM":"Tricycle"
}

VALID_PREFIXES = set(PLATE_MAP.keys())
PLATE_TYPE_MAP = {
    "ATM":"Military", "CD":"Diplomatic", "CC":"Diplomatic","Z":"Military",
    "KDN":"Special",  "VIP":"Special", "BM":"Tricycle"
}

# =========================
# SG Plate Detection Logic
# =========================

SG_PREFIXES = ["S", "E", "F", "G", "P", "Q"]

def is_singapore_plate(text):
    text = re.sub(r'[^A-Z0-9]', '', text.upper())

    # standard SG plate（suffix optional）
    if re.match(r"^S[A-Z]{1,2}\d{1,4}[A-Z]?$", text):
        return True

    # fallback：start with "S" but not Malaysia prefix
    if text.startswith("S"):
        prefix = re.match(r"^[A-Z]+", text)
        if prefix:
            prefix = prefix.group()
            if prefix not in VALID_PREFIXES:
                return True

    return False

def identify_state(raw_text):
    text = re.sub(r"[^A-Z0-9]", "", raw_text.upper().strip())

    if not text:
        return "Unknown", "Unknown"

    if is_singapore_plate(text):
        return "Singapore", "International"

    for special in ["ATM", "KDN", "VIP", "CD", "CC", "Z"]:
        if text.startswith(special):
            return PLATE_MAP.get(special), PLATE_TYPE_MAP.get(special, "Special")

    m = re.match(r"^([A-Z]+)", text)
    if m:
        prefix = m.group(1)

        for length in range(min(len(prefix), 3), 0, -1):
            c = prefix[:length]

            if c in PLATE_MAP:
                return PLATE_MAP[c], PLATE_TYPE_MAP.get(c, "Standard")

    return "Unknown", "Standard"

    prefix = m.group(1)
    for length in [3, 2, 1]:
        c = prefix[:length] if len(prefix) >= length else prefix
        if c in PLATE_MAP:
            return PLATE_MAP[c], PLATE_TYPE_MAP.get(c, "Standard")
    return "Unknown", "Standard"

print("=== Cell 4: State ID test ===")
for plate, exp in [
    ("WWB8886", "WP Kuala Lumpur"), ("VAT6458", "Selangor"),
    ("VCG7127", "Selangor"),        ("BNE520",  "Selangor"),
    ("T5944",   "Terengganu"),      ("ATM123",  "Military"),
]:
    s, _ = identify_state(plate)
    print(f"  {'OK  ' if s==exp else 'FAIL'}  {plate} -> {s}")
print("Cell 4 done")
print("=== SG TEST ===")
for plate in ["SNG1193S", "SBA1234A", "WAB1234"]:
    print(plate, "->", identify_state(plate))

=== Cell 4: State ID test ===
  OK    WWB8886 -> WP Kuala Lumpur
  OK    VAT6458 -> Selangor
  OK    VCG7127 -> Selangor
  OK    BNE520 -> Selangor
  OK    T5944 -> Terengganu
  OK    ATM123 -> Military
Cell 4 done
=== SG TEST ===
SNG1193S -> ('Singapore', 'International')
SBA1234A -> ('Singapore', 'International')
WAB1234 -> ('WP Kuala Lumpur', 'Standard')


In [ ]:
#===================
# Cell 5 - OCR
#===================

# ONLY safe prefix swaps (no digits here)
LOOKALIKES = {
    "N": ["W"],
    "K": ["W"],
    "H": ["W", "M"],
    "M": ["W"],
    "W": ["N", "K", "M"],
}

def fix_prefix(prefix):
    """Fix ONLY if prefix is invalid (no guessing)"""
    if prefix in VALID_PREFIXES:
        return prefix

    queue = deque([(prefix, 0)])
    visited = {prefix}

    while queue:
        candidate, depth = queue.popleft()
        if depth >= 2:
            continue

        for i, char in enumerate(candidate):
            for rep in LOOKALIKES.get(char, []):
                new = candidate[:i] + rep + candidate[i+1:]

                if new in visited:
                    continue
                visited.add(new)

                if new in VALID_PREFIXES:
                    print(f"[prefix fix] {prefix} -> {new}")
                    return new

                queue.append((new, depth + 1))

    return prefix


def fix_ocr_errors(text, ocr_conf=1.0):
    text = re.sub(r"[^A-Z0-9]", "", text.upper())
    if len(text) < 2:
        return ""

    d2l = {"0":"O","1":"I","5":"S","8":"B","2":"Z","6":"G"}
    l2d = {"O":"0","D":"0","I":"1","L":"1","S":"5","Z":"2","G":"6"}

    digit_match = re.search(r"\d", text)
    if not digit_match:
        return ""

    raw_prefix = text[:digit_match.start()]
    rest       = text[digit_match.start():]

    # =========================
    # Singapore plate
    # =========================
    if is_singapore_plate(text):
        prefix = "".join(d2l.get(c, c) for c in raw_prefix)

        trail = re.search(r"[A-Z]+$", rest)
        if trail:
            suffix = trail.group()
            number_part = rest[:trail.start()]
        else:
            suffix = ""
            number_part = rest

        number_part = "".join(l2d.get(c, c) for c in number_part)

        if not number_part:
            return ""

        #  SG：if don't have suffix but the last alphabet similar then fix
        if not suffix and len(number_part) >= 3:
            last = number_part[-1]
            if last in ["0","5","1","8"]:
                digit_to_letter = {"5":"S","1":"I","8":"B","0":"Q"}
                corrected = digit_to_letter[last]
                print(f"[SG suffix fix] {last} -> {corrected}")
                suffix = corrected
                number_part = number_part[:-1]

        return f"{prefix} {number_part}" + (f" {suffix}" if suffix else "")

    # =========================
    # Malaysia plate
    # =========================
    raw_prefix = "".join(d2l.get(c, c) for c in raw_prefix)[:3]

    if raw_prefix in VALID_PREFIXES and ocr_conf >= 0.80:
        fixed_prefix = raw_prefix
    else:
        fixed_prefix = fix_prefix(raw_prefix)

    trail = re.search(r"[A-Z]+$", rest)
    if trail:
        suffix = trail.group()
        number_part = rest[:trail.start()]
    else:
        suffix = ""
        number_part = rest

    number_part = "".join(l2d.get(c, c) for c in number_part)

    if not number_part:
        return ""

    # =========================
    # safe suffix repair
    # =========================
    if not suffix:
        if len(number_part) >= 3:
            last = number_part[-1]

            if last in ["0","5"]:
                digit_to_letter = {
                    "5": "S",
                    "0":"Q",
                }

                corrected = digit_to_letter[last]

                if fixed_prefix in VALID_PREFIXES:
                    print(f"[MY suffix fix] {last} -> {corrected}")
                    suffix = corrected
                    number_part = number_part[:-1]

    parts = [fixed_prefix, number_part]
    if suffix:
        parts.append(suffix)

    return " ".join(parts)


def preprocess_plate(plate_img):
    """Multi-preprocess + pick best OCR result (dynamic)"""
    if plate_img is None or plate_img.size == 0:
        return None, "", 0.0

    h, w = plate_img.shape[:2]
    if h < 15 or w < 40:
        return None, "", 0.0

    plate_img = cv2.resize(
        plate_img,
        (600, max(60, int(h * 600 / w))),
        interpolation=cv2.INTER_CUBIC
    )

    gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)

    clahe = cv2.createCLAHE(2.0, (8, 8))
    enhanced = clahe.apply(gray)

    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    sharp = cv2.filter2D(gray, -1, kernel)

    _, otsu = cv2.threshold(enhanced, 0, 255,
                            cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu_inv = cv2.bitwise_not(otsu)

    def run_ocr(img):
        try:
            res = ocr.readtext(
                img,
                allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
                detail=1,
                paragraph=False,
                text_threshold=0.5,
                low_text=0.3
            )
            if res:
                txt = "".join(r[1] for r in res).upper()
                conf = sum(r[2] for r in res)/len(res)
                return txt, conf
        except:
            pass
        return "", 0.0

    adaptive = cv2.adaptiveThreshold(
        enhanced, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 15, 8
    )
    candidates = [plate_img, sharp, enhanced, otsu, otsu_inv,adaptive]

    results = []
    for img in candidates:
        txt, conf = run_ocr(img)
        if txt:
            results.append((txt, conf))

    if not results:
        return plate_img, "", 0.0

    best_text, best_conf = max(results, key=lambda x: x[1])
    return plate_img, best_text, best_conf

def ocr_with_recovery(crop_bgr):
    if crop_bgr is None or crop_bgr.size == 0:
        return "", 0.0

    # use original logic get the text
    _, raw_text, conf = preprocess_plate(crop_bgr)

    # handle two line car plate
    lines = [line.strip() for line in raw_text.split('\n') if line.strip()]

    order_fixed = False

    if len(lines) >= 2:
        # if twoline, check is the 1 line is alphabet and 2 line is numbers
        first_line_is_digit = any(char.isdigit() for char in lines[0])
        second_line_is_alpha = any(char.isalpha() for char in lines[1])

        if first_line_is_digit and second_line_is_alpha:
            # change sequence
            raw_text = f"{lines[1]} {lines[0]}"
            print(f"[order_fix] change sequence: {raw_text}")

    # handle similar character recognition errors
    corrected = fix_ocr_errors(raw_text, conf)

    match = re.match(r'^(\d+)([A-Z]+)$', corrected.replace(" ", ""))
    if match:
        corrected = f"{match.group(2)} {match.group(1)}"
        print(f"[flip_fix] change the order: {corrected}")

    # if raw_text != corrected and "[order_fix]" not in locals():
    if raw_text != corrected and not order_fixed:
        print(f"[fix] {raw_text} -> {corrected}")

    return corrected, conf


def plate_score(text, ocr_conf):
    text_clean = re.sub(r'[^A-Z0-9]', '', text.upper())

    # if too short
    if len(text_clean) < 5:
        return 0

    score = ocr_conf

    # must have numbers
    if any(c.isdigit() for c in text_clean):
        score += 0.2
    else:
        return 0

    # prefix
    prefix_match = re.match(r'^([A-Z]+)', text_clean)
    if prefix_match:
        prefix = prefix_match.group(1)
        if prefix in VALID_PREFIXES:
            score += 0.3
        elif is_singapore_plate(text_clean):
            score += 0.2
        else:
            score -= 0.3

    # length
    if 5 <= len(text_clean) <= 8:
        score += 0.2

    # prevent OCR noise
    if len(set(text_clean)) >= 3:
        score += 0.1

    return min(score, 1.0)

print("CELL 5 FINAL READY")


CELL 5 FINAL READY


In [ ]:
# =========================
# CELL 6 — Plate detection
# =========================

class MalaysianPlateDetector:
    """traditional CV detector"""

    def __init__(self, minAR=1.2, maxAR=6.0):
        self.minAR = minAR
        self.maxAR = maxAR

    def detect(self, image):
        """Morphology + Sobel + Contour Filtering"""
        if isinstance(image, str):
            img = cv2.imread(image)
        else:
            img = image.copy()

        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # Morphology(single line)
        rectKern = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 5))
        tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, rectKern)

        # Kernel(two line)
        rectKernLarge = cv2.getStructuringElement(cv2.MORPH_RECT, (13, 13))
        tophatLarge = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, rectKernLarge)

        # combine
        tophat_combined = cv2.addWeighted(tophat, 0.6, tophatLarge, 0.4, 0)

        # Sobel edge detect
        gradX = cv2.Sobel(tophat_combined, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=3)
        gradX = np.absolute(gradX)
        (minVal, maxVal) = (np.min(gradX), np.max(gradX))
        if maxVal - minVal > 0:
            gradX = 255 * ((gradX - minVal) / (maxVal - minVal))
        gradX = gradX.astype("uint8")

        #Gaussian Blur + Closing Operation
        gradX = cv2.GaussianBlur(gradX, (5, 5), 0)

        closed1 = cv2.morphologyEx(gradX, cv2.MORPH_CLOSE, rectKern)
        closed2 = cv2.morphologyEx(gradX, cv2.MORPH_CLOSE, rectKernLarge)
        closed = cv2.addWeighted(closed1, 0.6, closed2, 0.4, 0)

        thresh = cv2.threshold(closed, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]

        thresh = cv2.erode(thresh, None, iterations=2)
        thresh = cv2.dilate(thresh, None, iterations=3)

        cnts = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cnts = imutils.grab_contours(cnts)

        valid_boxes = []
        for cnt in cnts:
            x, y, bw, bh = cv2.boundingRect(cnt)

            if bw < 40 or bh < 12:
                continue

            aspect = bw / bh
            area_ratio = (bw * bh) / (w * h)

            if (self.minAR <= aspect <= self.maxAR and 0.001 < area_ratio < 0.25):
                # dynamic padding
                pad_x = int(bw * 0.1)
                pad_y = int(bh * 0.25)
                x1 = max(0, x - pad_x)
                y1 = max(0, y - pad_y)
                x2 = min(w, x + bw + pad_x)
                y2 = min(h, y + bh + pad_y)
                valid_boxes.append((x1, y1, x2 - x1, y2 - y1))

        valid_boxes = self.merge_boxes(valid_boxes)

        valid_boxes.sort(key=lambda b: b[2]*b[3], reverse=True)
        return valid_boxes[:3]

    def merge_boxes(self, boxes, vertical_gap_max=40, horizontal_overlap_min=0.3):
        """Merge vertically adjacent boxes (used for two-line license plates)"""
        if len(boxes) < 2:
            return boxes

        merged = []
        used = [False] * len(boxes)

        for i, (x1, y1, w1, h1) in enumerate(boxes):
            if used[i]:
                continue

            x1_end = x1 + w1
            y1_end = y1 + h1
            current = [x1, y1, x1_end, y1_end]

            for j, (x2, y2, w2, h2) in enumerate(boxes):
                if i == j or used[j]:
                    continue

                x2_end = x2 + w2
                y2_end = y2 + h2

                # Check horizontal overlap
                x_overlap = max(x1, x2) < min(x1_end, x2_end)
                overlap_ratio = (min(x1_end, x2_end) - max(x1, x2)) / min(w1, w2)

                # Vertical distance
                vertical_gap = abs(y2 - y1_end)

                if x_overlap and overlap_ratio > horizontal_overlap_min and vertical_gap < vertical_gap_max:
                    current = [min(current[0], x2), min(current[1], y2),
                              max(current[2], x2_end), max(current[3], y2_end)]
                    used[j] = True

            merged.append((current[0], current[1], current[2]-current[0], current[3]-current[1]))
            used[i] = True

        return merged


def detect_plates(image_input):
    """Main detection function"""
    if isinstance(image_input, str):
        img_bgr = cv2.imread(image_input)
    else:
        img_bgr = image_input.copy()

    if img_bgr is None:
        return None, []

    detector = MalaysianPlateDetector(minAR=1.0, maxAR=6.0)

    # try original image
    boxes = detector.detect(img_bgr)

    # If not detected, try detection after image enhancement
    if len(boxes) == 0:
        # Enhance contrast
        lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        l_enhanced = clahe.apply(l)
        enhanced = cv2.merge([l_enhanced, a, b])
        enhanced = cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)
        boxes = detector.detect(enhanced)

    display_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    detections = []

    for (x, y, w, h) in boxes:
        x, y = max(0, x), max(0, y)
        w = min(display_rgb.shape[1] - x, w)
        h = min(display_rgb.shape[0] - y, h)

        if w < 40 or h < 15:
            continue

        crop_bgr = img_bgr[y:y+h, x:x+w]

        # try various OCR preprocessing methods
        plate_text, ocr_conf = ocr_with_recovery(crop_bgr)

        if not plate_text or ocr_conf < 0.5:
            gray = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2GRAY)

            thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                          cv2.THRESH_BINARY, 11, 2)
            thresh_bgr = cv2.cvtColor(thresh, cv2.COLOR_GRAY2BGR)
            text2, conf2 = ocr_with_recovery(thresh_bgr)
            if conf2 > ocr_conf:
                plate_text, ocr_conf = text2, conf2

        final_conf = plate_score(plate_text, ocr_conf)

        if final_conf < 0.9:
            continue

        state, plate_type = identify_state(plate_text)

        aspect = w / h
        if state == "Singapore":
            color = (0, 165, 200) # blue

        elif aspect < 2.2:
            color = (255, 100, 0) # orange

        else:
            color = (0, 255, 0) # green

        cv2.rectangle(display_rgb, (x, y), (x+w, y+h), color, 3)

        label = f"{plate_text} | {state}"
        font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2
        (lw, lh), base = cv2.getTextSize(label, font, scale, thick)

        label_y = y - 8
        if label_y - lh < 0:
            label_y = y + h + lh + 5

        cv2.rectangle(display_rgb, (x, label_y - lh - 4), (x + lw + 8, label_y + 4), color, -1)
        cv2.putText(display_rgb, label, (x + 4, label_y), font, scale, (0, 0, 0), thick, cv2.LINE_AA)

        detections.append({
            "plate_text": plate_text,
            "state": state,
            "plate_type": plate_type,
            "confidence": round(final_conf, 3),
            "ocr_conf": round(ocr_conf, 3),
            "bbox": [x, y, x+w, y+h]
        })

    return display_rgb, detections


print("Cell 6 done loading")

Cell 6 done loading


In [ ]:
# ====================
# CELL 7 — Gradio GUI
# ====================
import gradio as gr
from PIL import Image as PILImage


def gradio_predict(pil_image):
    if pil_image is None:
        return None, "Please upload an image."
    img_bgr   = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
    annotated, detections = detect_plates(img_bgr)
    if annotated is None:
        return None, "Error processing image."
    if not detections:
        text_out = "No plate detected.\n\nTips:\n  Make sure plate is clearly visible\n  Try a closer image"
    else:
        lines = []
        for i, d in enumerate(detections, 1):
            lines += [
                "="*35,
                f"  PLATE #{i}",
                "="*35,
                f"  Number     : {d['plate_text']}",
                f"  State      : {d['state']}",
                f"  Type       : {d['plate_type']}",
                f"  Confidence : {d['confidence']:.0%}",
                f"  OCR conf   : {d['ocr_conf']:.0%}",
                "",
            ]
        text_out = "\n".join(lines)
    return PILImage.fromarray(annotated), text_out


with gr.Blocks(title="Malaysian LPR System", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # Malaysian License Plate Recognition System
    **CT036-3-IPPR Group Assignment**
    """)
    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(label="Upload Image", type="pil", height=300)
            run_btn   = gr.Button("Detect & Identify", variant="primary", size="lg")
        with gr.Column(scale=1):
            img_output = gr.Image(label="Detection Result", type="pil", height=300)
            txt_output = gr.Textbox(label="Recognition Results", lines=14)
    run_btn.click(fn=gradio_predict, inputs=img_input, outputs=[img_output, txt_output])

app.launch(share=True, debug=False)

/tmp/ipykernel_4609/690064982.py:35: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Malaysian LPR System", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc84bab39f43c8feb4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
# =========================
# CELL 8 — DIAGNOSTIC TOOL
# ========================

# Widget support
try:
    import ipywidgets as widgets
    from IPython.display import display
    WIDGETS_AVAILABLE = True
except:
    print("Widgets not available → use manual path")
    WIDGETS_AVAILABLE = False


def diagnose_plate(image_path):

    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        print("Cannot load image")
        return

    # Use detect_plates() directly — no separate boxes variable
    ann_img, detections = detect_plates(img_bgr)

    if not detections:
        print("No plate detected")
        # Still show the annotated image
        plt.figure(figsize=(8, 6))
        plt.imshow(ann_img)
        plt.title("No plate detected")
        plt.axis("off")
        tmp_fig = "/tmp/diag_nodet.png"
        plt.savefig(tmp_fig, dpi=100, bbox_inches="tight")
        plt.close()
        ipdisplay(IPImage(tmp_fig))
        return

    print(f"\nFound {len(detections)} plate(s)\n")

    for i, det in enumerate(detections):
        print("=" * 50)
        print(f"PLATE #{i+1}")
        print("=" * 50)

        # Get crop from bounding box stored in detection
        x1, y1, x2, y2 = det.get("bbox", [0, 0, 50, 50])
        crop = img_bgr[max(0,y1-10):min(img_bgr.shape[0],y2+10),
                       max(0,x1-10):min(img_bgr.shape[1],x2+10)]

        if crop.size == 0:
            crop = img_bgr   # fallback to full image

        # Preprocessing stages
        gray     = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        clahe    = cv2.createCLAHE(2.0, (8,8)).apply(gray)
        blur     = cv2.bilateralFilter(clahe, 11, 100, 100)
        edges    = cv2.Canny(blur, 50, 150)
        _, otsu  = cv2.threshold(clahe, 0, 255,
                                  cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        otsu_inv = cv2.bitwise_not(otsu)
        adaptive = cv2.adaptiveThreshold(clahe, 255,
                       cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                       cv2.THRESH_BINARY, 15, 8)

        stages = {
            "Original": crop,
            "Gray":     gray,
            "CLAHE":    clahe,
            "Blur":     blur,
            "Edges":    edges,
            "Otsu":     otsu,
            "Otsu Inv": otsu_inv,
            "Adaptive": adaptive,
        }

        fig, axes = plt.subplots(1, len(stages), figsize=(22, 3))
        best_stage, best_text, best_conf = "", "", 0.0

        for ax, (name, img) in zip(axes, stages.items()):
            disp = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img.ndim == 3 else img
            ax.imshow(disp, cmap="gray" if img.ndim == 2 else None)

            raw, conf = "", 0.0
            try:
                res = ocr.readtext(img,
                    allowlist="ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789",
                    detail=1, paragraph=False)
                if res:
                    raw  = "".join(r[1] for r in res).upper()
                    conf = sum(r[2] for r in res) / len(res)
            except:
                pass

            if conf > best_conf:
                best_conf, best_text, best_stage = conf, raw, name

            ax.set_title(f"{name}\n{raw}\n({conf:.2f})", fontsize=8)
            ax.axis("off")

        plt.suptitle(f"Plate #{i+1} — Preprocessing Stages", fontsize=11)
        plt.tight_layout()
        tmp_fig = f"/tmp/diag_plate_{i}.png"
        plt.savefig(tmp_fig, dpi=100, bbox_inches="tight")
        plt.close()
        ipdisplay(IPImage(tmp_fig))

        print(f"Final Plate    : {det['plate_text']}")
        print(f"State          : {det['state']}")
        print(f"OCR conf       : {det['ocr_conf']:.2f}")
        print(f"Best stage     : {best_stage} ({best_conf:.2f})")
        print()


print("diagnose_plate() ready")

# ============================================================
# INPUT (UPLOAD UI)
# ============================================================

if WIDGETS_AVAILABLE:

    upload = widgets.FileUpload(accept="image/*", multiple=False)
    run_btn = widgets.Button(description="Run Diagnose", button_style="primary")
    output = widgets.Output()

    def on_click(b):
        output.clear_output()
        with output:
            if not upload.value:
                print("Please upload an image first.")
                return

            uploaded_file = list(upload.value.values())[0]
            ext = os.path.splitext(uploaded_file["metadata"]["name"])[1]

            tmp = tempfile.NamedTemporaryFile(suffix=ext, delete=False)
            tmp.write(uploaded_file["content"])
            tmp.close()

            diagnose_plate(tmp.name)
            os.unlink(tmp.name)

    run_btn.on_click(on_click)
    display(upload, run_btn, output)

else:
    print("\nManual usage:")
    print("diagnose_plate('image.jpg')")

diagnose_plate() ready


FileUpload(value={}, accept='image/*', description='Upload')

Button(button_style='primary', description='Run Diagnose', style=ButtonStyle())

Output()